# LLM-as-a-Judge: independent failure detection and calibration

Can a judge identify **Failure to interpret indirect or underspecified requests**
from a user request and the router's prediction?

The judge receives only `USER_INPUT`, `final_intent`, and a fixed routing policy.
It never sees the target intent, pass/fail, human comments, failure categories,
or reference Boolean. Every selected trace is judged, including correct routes.

We compare **GPT-5** and **GPT-6 Astra** on the same development cases, excluding the examples already shown in the prompt.
The router uses `gpt-4o-mini`. Larger judges are candidates, not automatically
reliable evaluators: agreement with the reference labels must be measured.

All 300 cases remain in the dataset. We reserve 150 for validation and use a
calibration panel drawn from the other 150. Six worked examples (three true and
three false) are included in the prompt. The remaining scoring panel contains
21 cases: nine reference positives and 12 negatives. This notebook runs the
calibration comparison by default; validation is a separate, disabled step.


The six-example prompt has not been run yet. Previous outputs were cleared to avoid presenting zero-shot scores as few-shot results. The earlier 24-case comparison is preserved in [the zero-shot metrics CSV](data/judge_calibration_runs/36bdad97a17d/calibration_metrics.csv). The updated prompt hash creates a separate result directory on the next run.


## 1. Setup

Run from the repository root or this notebook's directory. Credentials are read
from the existing `.env`; values are never displayed. Model requests use the
Responses API, a JSON schema, and the same reasoning effort. We omit temperature:
these reasoning configurations do not support it, and repeated runs can vary.


In [13]:
# Import the tools for loading data, calling the judges, and measuring agreement.
import hashlib
import importlib
import json
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import StratifiedGroupKFold

# Find the project files whether the notebook runs from the repo root or its own folder.
BASE_DIR = Path.cwd()
if (BASE_DIR / 'routing_agent_evals').is_dir():
    BASE_DIR = BASE_DIR / 'routing_agent_evals'
assert (BASE_DIR / 'prompts/llm_as_judge_prompt.py').is_file()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from prompts import llm_as_judge_prompt as judge_prompts
# Pick up prompt edits when rerunning cells in the same notebook kernel.
importlib.reload(judge_prompts)
INDIRECT_OR_UNDERSPECIFIED_FAILURE_PROMPT = judge_prompts.INDIRECT_OR_UNDERSPECIFIED_FAILURE_PROMPT
FEW_SHOT_CASE_IDS = judge_prompts.FEW_SHOT_CASE_IDS

# Load the API key from the local .env file without replacing existing environment settings.
load_dotenv(BASE_DIR / '.env', override=False)
# Keep this comparison local apart from the requested OpenAI inference calls.
os.environ['LANGSMITH_TRACING'] = 'false'
os.environ['LANGCHAIN_TRACING_V2'] = 'false'
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in routing_agent_evals/.env'


## 2. Load the reference dataset

`Indirect_or_Underspecified_Failure` is the reference answer for the judge.
`true` means this particular failure is present; `false` can include other routing
failures. Targets and annotations are available for offline sampling and review,
but will not enter any model request.

**Reference limitations:** These labels are provisional. Some annotations have
unresolved capability or ambiguity boundaries, and passing rows were assigned
false from target agreement. A disagreement can expose a judge error or a
reference/policy problem. Do not silently relabel cases to improve a score.


In [14]:
# Load all annotated traces; reference labels are used for sampling and scoring only.
DATASET_PATH = BASE_DIR / 'data/llm_as_judge_indirect_underspecified.csv'
LABEL = 'Indirect_or_Underspecified_Failure'
df = pd.read_csv(DATASET_PATH, keep_default_na=False)

def parse_boolean(value):
    # Convert CSV true/false values to Booleans; reject unexpected labels instead of guessing.
    normalized = str(value).strip().lower()
    if normalized not in {'true', 'false'}:
        raise ValueError(f'Invalid Boolean label: {value!r}')
    return normalized == 'true'

# Check row identities and confirm pass/fail matches the programmatic intent comparison.
df[LABEL] = df[LABEL].map(parse_boolean)
assert len(df) == 300 and df.TEST_ID.is_unique
assert df['intent_match'].eq(
    df['TARGET_INTENT'].eq(df['final_intent']).map({True: 'pass', False: 'fail'})
).all()
# Fingerprint this dataset version and show the accuracy of always predicting false.
DATASET_SHA256 = hashlib.sha256(DATASET_PATH.read_bytes()).hexdigest()
display(df[LABEL].value_counts().rename_axis('Reference label').to_frame('Count'))
print(f'Always-false baseline accuracy: {(~df[LABEL]).mean():.1%}')


,Count
Reference label,
False,277
True,23


Always-false baseline accuracy: 92.3%


## 3. Reserve validation before calibration

Stratify using three offline groups: selected failures, other routing failures,
and passing cases. Keep known paraphrase families together. Their IDs below are
used only to keep related examples on one side of the split, never in the prompt.

The fixed split contains 150 development cases (12 positives) and 150 validation
cases (11 positives). Known families have been grouped conservatively; this is
validation reserved from subsequent tuning, not a pristine new production sample.


In [15]:
# Keep known paraphrases together so similar requests do not cross the development/validation split.
# The fixed seed makes the split and calibration sample reproducible.
SEED = 42
KNOWN_PARAPHRASE_GROUPS = [['TEST_040', 'TEST_045', 'TEST_046', 'TEST_098', 'TEST_105', 'TEST_168', 'TEST_171', 'TEST_255', 'TEST_294'], ['TEST_038', 'TEST_042', 'TEST_158', 'TEST_170', 'TEST_174', 'TEST_196', 'TEST_199', 'TEST_202', 'TEST_207', 'TEST_236', 'TEST_241', 'TEST_246'], ['TEST_043', 'TEST_205'], ['TEST_016', 'TEST_086', 'TEST_095', 'TEST_223', 'TEST_226'], ['TEST_021', 'TEST_107', 'TEST_211', 'TEST_212', 'TEST_213', 'TEST_215', 'TEST_218', 'TEST_219', 'TEST_220', 'TEST_221', 'TEST_222'], ['TEST_064', 'TEST_216', 'TEST_217'], ['TEST_063', 'TEST_225', 'TEST_227'], ['TEST_031', 'TEST_032', 'TEST_138', 'TEST_141', 'TEST_145', 'TEST_151', 'TEST_247', 'TEST_250', 'TEST_254', 'TEST_257'], ['TEST_112', 'TEST_235', 'TEST_243', 'TEST_258'], ['TEST_088', 'TEST_237', 'TEST_238', 'TEST_242', 'TEST_251', 'TEST_266', 'TEST_283', 'TEST_289'], ['TEST_121', 'TEST_129', 'TEST_276'], ['TEST_025', 'TEST_143', 'TEST_148'], ['TEST_009', 'TEST_069'], ['TEST_162', 'TEST_240'], ['TEST_034', 'TEST_068', 'TEST_161', 'TEST_239', 'TEST_248', 'TEST_261']]
group_lookup = {case_id: group[0] for group in KNOWN_PARAPHRASE_GROUPS for case_id in group}
assert len(group_lookup) == sum(map(len, KNOWN_PARAPHRASE_GROUPS))
assert set(group_lookup).issubset(set(df.TEST_ID))
df['paraphrase_group'] = df.TEST_ID.map(group_lookup).fillna(df.TEST_ID)

# These labels control offline sampling only, not inference or a correctness gate.
df['sampling_group'] = 'passing'
df.loc[df.intent_match.eq('fail'), 'sampling_group'] = 'other_failure'
df.loc[df[LABEL], 'sampling_group'] = 'selected_failure'
# Create two sets with a similar mix of failures and passes while keeping paraphrase groups intact.
splitter = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=SEED)
development_idx, validation_idx = next(
    splitter.split(df, df.sampling_group, groups=df.paraphrase_group)
)
df['split'] = 'validation'
df.loc[development_idx, 'split'] = 'development'
development = df.loc[development_idx].copy()
validation = df.loc[validation_idx].copy()
assert set(development.paraphrase_group).isdisjoint(validation.paraphrase_group)
assert set(development.TEST_ID).isdisjoint(validation.TEST_ID)

# Build a 24-case development sample: 12 selected failures, six other failures, and six passing routes.
# Passing cases include related paraphrases and ordinary requests; prompt examples are removed below.
positive_cases = development[development[LABEL]].sample(n=12, random_state=SEED)
other_failures = development[development.sampling_group.eq('other_failure')].sample(n=6, random_state=SEED)
passing_cases = development[development.sampling_group.eq('passing')]
related_passes = passing_cases[passing_cases.TEST_ID.isin(group_lookup)].sample(n=3, random_state=SEED)
ordinary_passes = passing_cases[~passing_cases.TEST_ID.isin(group_lookup)].sample(n=3, random_state=SEED)
calibration = pd.concat([positive_cases, other_failures, related_passes, ordinary_passes])
calibration = calibration.sample(frac=1, random_state=SEED).reset_index(drop=True)
assert len(calibration) == 24 and calibration[LABEL].sum() == 12
assert set(calibration.TEST_ID).issubset(development.TEST_ID)

# Demonstration answers are visible to the model; do not grade it on those cases.
assert FEW_SHOT_CASE_IDS.issubset(set(development.TEST_ID))
assert FEW_SHOT_CASE_IDS.isdisjoint(set(validation.TEST_ID))
calibration = calibration[~calibration.TEST_ID.isin(FEW_SHOT_CASE_IDS)].reset_index(drop=True)
assert len(calibration) == 21 and calibration[LABEL].sum() == 9

# Save each case's split and role, then display the group counts for inspection.
df['in_calibration'] = df.TEST_ID.isin(calibration.TEST_ID)
df['in_prompt'] = df.TEST_ID.isin(FEW_SHOT_CASE_IDS)
SPLIT_PATH = BASE_DIR / 'data/judge_indirect_split.csv'
df[['TEST_ID', 'split', 'in_calibration', 'in_prompt', 'paraphrase_group']].to_csv(SPLIT_PATH, index=False)
display(pd.crosstab(df.sampling_group, df.split).assign(Total=lambda x: x.sum(axis=1)))
display(calibration.sampling_group.value_counts().to_frame('Calibration cases'))

print(f'Scored calibration cases: {len(calibration)}; always-false baseline: {(~calibration[LABEL]).mean():.1%}')


split,development,validation,Total
sampling_group,,,
other_failure,14,15,29
passing,124,124,248
selected_failure,12,11,23


,Calibration cases
sampling_group,
selected_failure,9
other_failure,6
passing,6


Scored calibration cases: 21; always-false baseline: 57.1%


## 4. Independent judge contract

The prompt in `prompts/llm_as_judge_prompt.py` contains the fixed routing policy
and the selected failure rubric. A recognizable intent can lack execution
details. Unsupported capabilities, clear polarity errors, and defensible
ambiguous routes are distinguished from this failure category.

The same system prompt and six worked examples are used for both models.
Examples show recognizable availability/rescheduling requests, a correctly
routed incomplete request, a polarity error outside this category, and a
correctly rejected named-patient operation. Their case IDs are retained only
locally to exclude demonstrations from scoring; IDs and targets do not enter
the prompt. No target intent or prior
correctness decision is supplied. The rationale is a short evidence explanation,
not a request for hidden chain-of-thought. Model reasoning and confidence from
the router are omitted in this baseline.


In [16]:
# Require a Boolean category label and a short explanation; reject extra fields or wrong types.
class JudgeOutputSchema(BaseModel):
    model_config = ConfigDict(extra='forbid', strict=True)
    label: bool = Field(description='Whether the selected indirect/underspecified interpretation failure is established.')
    reasoning: str = Field(description='One or two short sentences citing the request and routing-policy boundary.')

def judge_input(user_query, predicted_intent):
    # Send only the request and predicted intent as JSON; never include targets or human labels.
    return json.dumps({'USER_INPUT': str(user_query), 'final_intent': str(predicted_intent)}, ensure_ascii=False)

# Keep the policy and examples in the system message, with each trace in a separate user message.
prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=INDIRECT_OR_UNDERSPECIFIED_FAILURE_PROMPT),
    ('human', '{case_json}'),
])
assert set(prompt.input_variables) == {'case_json'}
display(json.loads(judge_input(calibration.iloc[0].USER_INPUT, calibration.iloc[0].final_intent)))


{'USER_INPUT': "I'm available on the 7th from 6 PM onwards.",
 'final_intent': 'UNBLOCK_SLOTS'}

## 5. Compare GPT-5 and GPT-6 Astra

Use the explicit model IDs `gpt-5` and `gpt-6-astra`, both with medium reasoning
and the same output budget. Model availability is checked before inference.
No alternate model is silently substituted. Returned model IDs and token usage
are saved with the results. Repeating an alias in the future is not a guarantee
of an identical model snapshot or identical predictions.


In [17]:
# Compare both models with the same reasoning settings and limit concurrent calls to four.
MODEL_IDS = ['gpt-5', 'gpt-6-astra']
MODEL_SETTINGS = {'reasoning': {'effort': 'medium'}, 'max_tokens': 4096}
MAX_WORKERS = 4
# Reuse saved judgments unless FORCE_RERUN is true; keep held-out validation off during prompt tuning.
FORCE_RERUN = False
RUN_VALIDATION = False

# Check account access to the exact requested models before making inference calls.
api = OpenAI(timeout=30, max_retries=0)
availability = []
for model_id in MODEL_IDS:
    try:
        available_model = api.models.retrieve(model_id)
        availability.append({'requested_model': model_id, 'available_model': available_model.id})
    except Exception as exc:
        raise RuntimeError(f'{model_id} is unavailable: {type(exc).__name__}; no substitute was selected.') from None
display(pd.DataFrame(availability))

# Build one prompt/model pipeline per judge, with the same structured JSON output contract.
judges = {}
for model_id in MODEL_IDS:
    model = init_chat_model(
        model_id, model_provider='openai', use_responses_api=True,
        timeout=180, max_retries=1, **MODEL_SETTINGS,
    )
    judges[model_id] = prompt | model.with_structured_output(
        JudgeOutputSchema.model_json_schema(), method='json_schema', strict=True, include_raw=True,
    )

# Give each prompt/settings/schema combination its own results folder to avoid reusing stale judgments.
PROMPT_SHA256 = hashlib.sha256(INDIRECT_OR_UNDERSPECIFIED_FAILURE_PROMPT.encode()).hexdigest()
RUN_SHA256 = hashlib.sha256(json.dumps(
    {'prompt': PROMPT_SHA256, 'settings': MODEL_SETTINGS, 'schema': JudgeOutputSchema.model_json_schema()}, sort_keys=True
).encode()).hexdigest()
RUN_DIR = BASE_DIR / 'data/judge_calibration_runs' / RUN_SHA256[:12]
RUN_DIR.mkdir(parents=True, exist_ok=True)
# Record the models, dataset version, and selected cases so the comparison can be reproduced.
(RUN_DIR / 'run_config.json').write_text(json.dumps({
    'models': MODEL_IDS, 'settings': MODEL_SETTINGS, 'prompt_sha256': PROMPT_SHA256,
    'dataset_sha256': DATASET_SHA256, 'split_seed': SEED,
    'calibration_ids': calibration.TEST_ID.tolist(),
    'few_shot_case_ids': sorted(FEW_SHOT_CASE_IDS),
    'validation_ids': validation.TEST_ID.tolist(),
    'judge_input_fields': ['USER_INPUT', 'final_intent'],
}, indent=2))
print(f'Result directory: {RUN_DIR.relative_to(BASE_DIR)}')


,requested_model,available_model
0,gpt-5,gpt-5
1,gpt-6-astra,gpt-6-astra


Result directory: data/judge_calibration_runs/ac02d58eb379


## 6. Run the same calibration cases through both judges

Each judge sees all 21 scoring traces, including passing routes. The three
positive demonstrations that overlapped the original 24-case panel are excluded. The runner has no access
to reference columns: we pass an input-only table into it. Successful responses
are cached by model, prompt, settings, and exact input. Changing the prompt or
input invalidates that cache. Set `FORCE_RERUN=True` to request fresh responses.

API or parsing errors remain unscored and are displayed explicitly. They are
never converted into false labels. The limited concurrency reduces waiting;
per-case latency is descriptive, not a controlled speed benchmark.


In [8]:
# Define how to judge one trace and run the same set of traces through both models.
def llm_as_judge_eval(user_query, predicted_intent, model_id):
    # Call one judge, validate its answer, and record the explanation, token usage, and latency.
    started = time.perf_counter()
    response = judges[model_id].invoke({'case_json': judge_input(user_query, predicted_intent)})
    parsed = response['parsed']
    # An invalid response is a failed call, not evidence that the selected failure is absent.
    if response.get('parsing_error') or parsed is None:
        raise ValueError('Judge output could not be parsed; this is not a false label.')
    parsed = JudgeOutputSchema.model_validate(parsed)
    raw = response['raw']
    usage = raw.usage_metadata or {}
    return {
        'judge_label': parsed.label, 'judge_reasoning': parsed.reasoning,
        'returned_model': raw.response_metadata.get('model_name', raw.response_metadata.get('model', model_id)),
        'input_tokens': usage.get('input_tokens', 0),
        'output_tokens': usage.get('output_tokens', 0),
        'latency_seconds': round(time.perf_counter() - started, 3),
        'error': '',
    }

def run_comparison(input_rows, subset_name):
    # Run every trace through every model, reusing cached answers and saving new results.
    # Accept only inference fields plus a tracking ID, and exclude examples already shown in the prompt.
    assert list(input_rows.columns) == ['TEST_ID', 'USER_INPUT', 'final_intent']
    assert input_rows.TEST_ID.is_unique
    assert FEW_SHOT_CASE_IDS.isdisjoint(set(input_rows.TEST_ID)), 'Exclude prompt examples from evaluation calls.'
    results = []
    jobs = []
    cache_dir = RUN_DIR / 'cache'
    cache_dir.mkdir(exist_ok=True)
    # Reuse a saved answer only when the prompt settings, model, and input all match.
    for model_id in MODEL_IDS:
        for row in input_rows.to_dict('records'):
            payload = judge_input(row['USER_INPUT'], row['final_intent'])
            fingerprint = hashlib.sha256((RUN_SHA256 + model_id + payload).encode()).hexdigest()
            cache_path = cache_dir / f'{fingerprint}.json'
            identity = {**row, 'model': model_id, 'subset': subset_name, 'input_sha256': fingerprint}
            if cache_path.exists() and not FORCE_RERUN:
                cached = json.loads(cache_path.read_text())
                results.append({**identity, **cached, 'cached': True})
            else:
                jobs.append((identity, cache_path))

    # Worker for one uncached judgment: save successes and record API/parse failures separately.
    def evaluate(identity, cache_path):
        try:
            result = llm_as_judge_eval(identity['USER_INPUT'], identity['final_intent'], identity['model'])
            cache_path.write_text(json.dumps(result, ensure_ascii=False))
        except Exception as exc:
            status = getattr(exc, 'status_code', None)
            result = {'judge_label': None, 'judge_reasoning': '', 'returned_model': '',
                      'input_tokens': 0, 'output_tokens': 0, 'latency_seconds': None,
                      'error': f'{type(exc).__name__}' + (f' (HTTP {status})' if status else '')}
        return {**identity, **result, 'cached': False}

    output_path = RUN_DIR / f'{subset_name}_predictions.csv'
    # Run pending calls concurrently and save progress as each call completes.
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [pool.submit(evaluate, identity, cache_path) for identity, cache_path in jobs]
        for completed, future in enumerate(as_completed(futures), start=1):
            result = future.result()
            results.append(result)
            # Checkpoint completed calls without exposing any credentials.
            pd.DataFrame(results).to_csv(output_path, index=False)
            if completed % 6 == 0 or completed == len(jobs) or result['error']:
                print(f'{subset_name}: {completed}/{len(jobs)} new calls complete; '
                      f'{sum(bool(r["error"]) for r in results)} errors', flush=True)

    # Verify there is exactly one result per model/case pair, then save the complete table.
    predictions = pd.DataFrame(results).sort_values(['model', 'TEST_ID']).reset_index(drop=True)
    assert len(predictions) == len(input_rows) * len(MODEL_IDS)
    assert not predictions.duplicated(['model', 'TEST_ID']).any()
    predictions.to_csv(output_path, index=False)
    return predictions


In [9]:
# Compare both judges on the development calibration sample, then show call counts and token usage.
# Send both passing and failing routes without revealing their reference labels.
calibration_inputs = calibration[['TEST_ID', 'USER_INPUT', 'final_intent']].copy()
calibration_predictions = run_comparison(calibration_inputs, 'calibration')
display(calibration_predictions.groupby('model').agg(
    cases=('TEST_ID', 'size'), errors=('error', lambda x: x.ne('').sum()),
    cached=('cached', 'sum'), input_tokens=('input_tokens', 'sum'),
    output_tokens=('output_tokens', 'sum'),
))


calibration: 6/42 new calls complete; 0 errors
calibration: 12/42 new calls complete; 0 errors
calibration: 18/42 new calls complete; 0 errors
calibration: 24/42 new calls complete; 0 errors
calibration: 30/42 new calls complete; 0 errors
calibration: 36/42 new calls complete; 0 errors
calibration: 42/42 new calls complete; 0 errors


,cases,errors,cached,input_tokens,output_tokens
model,,,,,
gpt-5,21,0,0,38181,17855
gpt-6-astra,21,0,0,38181,1889


## 7. Measure agreement with the hidden reference labels

Reference labels are joined **after inference**. Positive-class precision asks
whether flagged cases match the reference; recall asks how many reference
positives were found. F1 summarizes their balance. Report the confusion counts
and coverage alongside percentages.

This panel deliberately contains more positives than the master dataset.
Its precision and accuracy do not estimate production performance. After
excluding demonstrations, its always-false baseline is 57.1% (12 of 21),
compared with 92.3% across the 300-case master dataset. Scores measure agreement with
provisional annotations and do not establish that every disagreement is a judge
mistake. Compare paired, successfully scored cases if any API calls failed.


In [10]:
# Measure agreement with human reference labels after inference; labels never enter the judge call.
def score_predictions(predictions, references, subset_name):
    # Join predictions to references and compare models on the same successfully judged cases.
    assert FEW_SHOT_CASE_IDS.isdisjoint(set(references.TEST_ID)), 'Prompt examples cannot be scored as evaluation cases.'
    scored = predictions.merge(references[['TEST_ID', LABEL]], on='TEST_ID', validate='many_to_one')
    # If either model failed on a case, exclude that case from both models' metric calculations.
    paired_ids = set(references.TEST_ID)
    for model_id in MODEL_IDS:
        paired_ids &= set(scored.loc[scored.model.eq(model_id) & scored.error.eq(''), 'TEST_ID'])
    summaries = []
    for model_id in MODEL_IDS:
        all_model_rows = scored[scored.model.eq(model_id)]
        matched = all_model_rows[all_model_rows.TEST_ID.isin(paired_ids)].copy()
        truth = matched[LABEL].map(parse_boolean).to_numpy(dtype=bool)
        predicted = matched.judge_label.map(parse_boolean).to_numpy(dtype=bool)
        # Count correct flags, false alarms, missed failures, and correct rejections of this category.
        tn, fp, fn, tp = (
            confusion_matrix(truth, predicted, labels=[False, True]).ravel()
            if len(matched) else (0, 0, 0, 0)
        )
        # Precision measures flag reliability; recall measures detection coverage; F1 combines them.
        precision, recall, f1, _ = precision_recall_fscore_support(
            truth, predicted, average='binary', zero_division=0
        ) if len(matched) else (float('nan'),) * 4
        summaries.append({
            'model': model_id, 'requested_cases': len(all_model_rows),
            'successfully_scored': int(all_model_rows.error.eq('').sum()),
            'paired_cases': len(matched), 'reference_positives': int(truth.sum()),
            'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
            'precision': precision, 'recall': recall, 'f1': f1,
            'accuracy': float((truth == predicted).mean()) if len(matched) else float('nan'),
            'always_false_accuracy': float((~truth).mean()) if len(matched) else float('nan'),
        })
        display(pd.DataFrame([[tn, fp], [fn, tp]],
            index=['Reference false', 'Reference true'],
            columns=[f'{model_id}: false', f'{model_id}: true']))
    # Save aggregate metrics and the row-level predictions/references needed to investigate disagreements.
    summary = pd.DataFrame(summaries).set_index('model')
    summary.to_csv(RUN_DIR / f'{subset_name}_metrics.csv')
    scored.to_csv(RUN_DIR / f'{subset_name}_alignment.csv', index=False)
    return scored, summary

# Score the calibration run and display the model comparison.
calibration_alignment, calibration_summary = score_predictions(
    calibration_predictions, calibration, 'calibration'
)
display(calibration_summary.round(3))


,gpt-5: false,gpt-5: true
Reference false,11,1
Reference true,3,6


,gpt-6-astra: false,gpt-6-astra: true
Reference false,11,1
Reference true,5,4


,requested_cases,successfully_scored,paired_cases,reference_positives,TP,FP,FN,TN,precision,recall,f1,accuracy,always_false_accuracy
model,,,,,,,,,,,,,
gpt-5,21,21,21,9,6,1,3,11,0.857,0.667,0.750,0.810,0.571
gpt-6-astra,21,21,21,9,4,1,5,11,0.800,0.444,0.571,0.714,0.571


## 8. Inspect disagreements before changing the prompt

For each disagreement, inspect the request, predicted intent, judge explanation,
human comment, and policy. Decide whether it reveals a rubric problem, a judge
mistake, or a reference-label/policy conflict. Resolve human-label issues through
review, not by accepting whichever model sounds persuasive.

Examples of known unresolved boundaries include time-selected individual
appointments and availability updates versus unblocking without calendar state.
The six prompt demonstrations come only from development and are excluded
from scoring. Any additional examples must follow the same rule. Keep
validation untouched.


In [11]:
# Add human notes after scoring to inspect where each judge differs from the reference label.
# These disagreements can expose judge mistakes or inconsistencies in the annotation contract.
review = calibration_alignment.merge(
    calibration[['TEST_ID', 'TARGET_INTENT', 'Human Validator Comments (Open Codes)', 'Failure Categories']],
    on='TEST_ID', validate='many_to_one',
)
disagreements = review[
    review.error.eq('') & review.judge_label.ne(review[LABEL])
].copy()
disagreements.to_csv(RUN_DIR / 'calibration_disagreements.csv', index=False)
pd.set_option('display.max_colwidth', 160)
display(disagreements[['model', 'TEST_ID', 'USER_INPUT', 'final_intent', LABEL,
                       'judge_label', 'judge_reasoning', 'Human Validator Comments (Open Codes)']])
# Show failed calls separately; they are not false positives or false negatives.
errors = calibration_predictions[calibration_predictions.error.ne('')]
if len(errors):
    display(errors[['model', 'TEST_ID', 'error']])


,model,TEST_ID,USER_INPUT,final_intent,Indirect_or_Underspecified_Failure,judge_label,judge_reasoning,Human Validator Comments (Open Codes)
9,gpt-5,TEST_202,Open up my schedule for this coming Sunday.,UNBLOCK_SLOTS,True,False,"'Open up my schedule for this coming Sunday' clearly implies restoring availability, which aligns with UNBLOCK_SLOTS. Even if it could also mean adding hour...","The conversational request to open Sunday's schedule is labeled as an availability update, but the model chose unblock. Calendar context is needed to distin..."
10,gpt-5,TEST_207,I'm available on the 7th from 6 PM onwards.,UNBLOCK_SLOTS,True,False,"'I'm available on the 7th from 6 PM onwards' implies making time available; per policy, this can reasonably map to UNBLOCK_SLOTS or UPDATE_SPECIAL_SLOTS wit...","'I'm available on the 7th from 6 PM onwards' states availability; the target expects an update, but the model chose unblock. The request does not establish ..."
11,gpt-5,TEST_228,I need to cancel one appointment today.,UNSUPPORTED_REQUEST,True,False,"“Cancel one appointment today” requests cancelling a single appointment, which the policy does not support (it only supports canceling all within a range). ...","The cancellation intent is clear, but the model marked the request unsupported. The target assumes individual cancellations are supported; that needs confir..."
18,gpt-5,TEST_254,I'm not coming in on the 26th.,CANCEL_APPOINTMENTS,False,True,'I'm not coming in on the 26th' implicitly requests making the doctor unavailable that day (block slots). Selecting CANCEL_APPOINTMENTS conflates unavailabi...,
30,gpt-6-astra,TEST_202,Open up my schedule for this coming Sunday.,UNBLOCK_SLOTS,True,False,"'Open up my schedule' reasonably means making the doctor available, which supports UNBLOCK_SLOTS. Without prior calendar state, changing working hours may a...","The conversational request to open Sunday's schedule is labeled as an availability update, but the model chose unblock. Calendar context is needed to distin..."
31,gpt-6-astra,TEST_207,I'm available on the 7th from 6 PM onwards.,UNBLOCK_SLOTS,True,False,"'I'm available on the 7th from 6 PM onwards' expresses availability, which can support UNBLOCK_SLOTS. Without calendar state distinguishing restored availab...","'I'm available on the 7th from 6 PM onwards' states availability; the target expects an update, but the model chose unblock. The request does not establish ..."
32,gpt-6-astra,TEST_228,I need to cancel one appointment today.,UNSUPPORTED_REQUEST,True,False,"“Cancel one appointment today” requests cancellation of an individual appointment, not all appointments within a time range. UNSUPPORTED_REQUEST respects th...","The cancellation intent is clear, but the model marked the request unsupported. The target assumes individual cancellations are supported; that needs confir..."
34,gpt-6-astra,TEST_239,I need to free up my morning.,UNSUPPORTED_REQUEST,True,False,"'Free up my morning' could mean canceling appointments, moving them, or blocking availability. Because the requested operation remains unclear and these eff...","The target expects cancellation, but the model marked “free up my morning” unsupported. The wording could mean cancel, move, or block appointments, so the i..."
36,gpt-6-astra,TEST_242,Push my schedule.,UNSUPPORTED_REQUEST,True,False,"'Push my schedule' does not clearly distinguish shifting appointments from changing working hours. Because the operation itself remains ambiguous, UNSUPPORT...","“Push my schedule” indicates rescheduling, but the model marked it unsupported. Clarify the affected date and how far to move the schedule without losing th..."
39,gpt-6-astra,TEST_254,I'm not coming in on the 26th.,CANCEL_APPOINTMENTS,False,True,"'I'm not coming in on the 26th' implicitly communicates unavailability, supporting BLOCK_SLOTS. CANCEL_APPOINTMENTS instead assumes cancellation of existing...",


## 9. Reserved validation after the rubric and prompt are frozen

`RUN_VALIDATION` is false for the initial calibration comparison. After reference
review and prompt development, set it true to run both judges on all 150 reserved
traces, including the 124 passing routes. The same input whitelist is used.

Do not use validation disagreements to keep tuning and then call the resulting
score an untouched test. With only 11 reference positives, a single changed
decision shifts recall by about nine percentage points. A fresh labeled sample
will be needed for a stronger claim about production performance.


In [ ]:
# Enable this only after choosing the prompt and judging rules, to evaluate the reserved cases.
# It uses the same input restrictions and scoring procedure as calibration.
if RUN_VALIDATION:
    validation_inputs = validation[['TEST_ID', 'USER_INPUT', 'final_intent']].copy()
    validation_predictions = run_comparison(validation_inputs, 'validation')
    validation_alignment, validation_summary = score_predictions(
        validation_predictions, validation, 'validation'
    )
    display(validation_summary.round(3))
else:
    print(f'{len(validation)} validation cases reserved; no validation inference calls made.')


150 validation cases reserved; no validation inference calls made.


The prompt, model settings, selected IDs, predictions, metrics, and disagreements
are saved together under the result directory. The original reference dataset
and earlier alignment result files are preserved.

References: [OpenAI evaluation guidance](https://developers.openai.com/api/docs/guides/evaluation-best-practices),
[GPT-5](https://developers.openai.com/api/docs/models/gpt-5),
[GPT-6 Astra](https://developers.openai.com/api/docs/models/gpt-6-astra), and
[Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs).
